In [1]:
import numpy as np
import pandas as pd

In [2]:
tx = pd.read_parquet("../data_generator/output/offline/transactions.parquet")

tx["hour"] = pd.to_datetime(tx["timestamp"]).dt.hour
print(tx["hour"].value_counts(normalize=True).sort_index())

print(tx["channel"].value_counts(normalize=True))

dup_rate = tx["transaction_id"].duplicated().mean()
print(f"Duplicate rate: {dup_rate:.4f}")  # kỳ vọng ~0.02

sample = tx.sample(20)
print(sample[["type", "old_balance", "amount", "new_balance"]])


print(tx["merchant_id"].isna().mean())
print(tx["counterparty_account_id"].isna().mean())  

hour
0     0.010532
1     0.010600
2     0.010493
3     0.010409
4     0.010306
5     0.010353
6     0.010417
7     0.116608
8     0.117659
9     0.117919
10    0.010488
11    0.010395
12    0.118157
13    0.010184
14    0.010517
15    0.010517
16    0.010495
17    0.010348
18    0.116792
19    0.117559
20    0.117907
21    0.010434
22    0.010358
23    0.010554
Name: proportion, dtype: float64
channel
app    0.672914
web    0.163844
atm    0.163242
Name: proportion, dtype: float64
Duplicate rate: 0.0196
            type  old_balance      amount  new_balance
305019   deposit  23652865.64   695070.83  24347936.47
2956    transfer  11809036.69  1052002.50  10757034.19
192111   payment  17535815.87   420881.62  17114934.25
345568  withdraw  14049442.94   929634.41  13119808.53
196643   payment  26839898.03  1753281.27  25086616.76
248378  transfer  20565278.02  1156186.95  19409091.07
104186   payment   5741092.29   907024.33   4834067.96
87064    deposit  13252539.01   917457.49  1416999

In [3]:
print(tx["status"].value_counts(normalize=True))

failed_check = tx[tx["status"] == "failed"]
print((failed_check["new_balance"] == failed_check["old_balance"]).mean()) 

non_deposit = tx[tx["type"] != "deposit"]
should_fail = non_deposit[non_deposit["amount"] > non_deposit["old_balance"]]
print((should_fail["status"] == "failed").mean())  

status
success    0.940816
failed     0.039699
pending    0.019485
Name: proportion, dtype: float64
1.0
1.0


In [4]:
merchants = pd.read_parquet("../data_generator/output/offline/merchants.parquet")
top5_ids = merchants.head(int(len(merchants)*0.05))["merchant_id"]  # cần chọn đúng top, không phải random sample
payment_tx = tx[tx["type"] == "payment"]
print(payment_tx["merchant_id"].isin(top5_ids).mean())  # kỳ vọng ~0.80, hiện tại nếu random đều sẽ ra ~0.05

0.7973052497364098
